# PotSpot: train the pothole model on a free Colab GPU

The model shipped in the repo (`models/pothole.pt`) was trained on a 2-core CPU: YOLO11n, 512 px, 35 epochs.
On a free T4 GPU you can train a bigger, sharper model in about 30 to 40 minutes.

1. **Runtime > Change runtime type > T4 GPU**
2. Run the cells in order.
3. Download `pothole.pt` at the end and put it in your project's `models/` folder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone --depth 1 https://github.com/SiddharthBhat120/POTSPOT.git
%cd POTSPOT
!pip install -q ultralytics albumentations

## 1. Dataset
Downloads the 1,243-photo pothole dataset (Chitale et al., IVCNZ 2020, MIT licence), checks every label,
keeps near-duplicate photos in the same split (57% of photos have a twin), and adds rain / motion-blur / dusk variants to the training split only.

In [ ]:
!python -m ml.data.download
!python -m ml.data.prepare
!python -m ml.data.augment_road

### Optional: add RDD2022 (India) potholes
RDD2022 has thousands of road photos from India and five other countries (class D40 = pothole).
Download `RDD2022_India.zip` from figshare into `/content`, then run the next cell. It converts the
VOC labels to YOLO, keeps only potholes, and merges them into the **train** split
(the test split stays the same, so scores stay comparable).

In [ ]:
# !python -m ml.data.rdd2022 /content/RDD2022_India.zip --countries India

## 2. Train
`yolo11s` at 640 px is a good step up from the CPU model. Patience stops early if validation stops improving.

In [ ]:
!python -m ml.train --model yolo11s.pt --imgsz 640 --epochs 100 --batch 32 --device 0 --workers 4

## 3. Evaluate on the held-out test split
Precision = how many boxes were real potholes. Recall = how many real potholes were found.
mAP50 = overall detection quality (1.0 is perfect). The failure images show what it still gets wrong.

In [ ]:
!python -m ml.evaluate --imgsz 640
from IPython.display import Image, display
display(Image('docs/eval/failures_missed.jpg', width=900))
display(Image('docs/eval/failures_false.jpg', width=900))

In [ ]:
from google.colab import files
files.download('models/pothole.pt')
files.download('docs/eval/metrics.json')